# R Programming Laboratory Assignment 2

## Lab 3: Control Flow for Data Cleaning
### Loops, Functions, and Error Handling in R

## Lab 4: Advanced Missing Data Handling
### NA, NULL, NaN, Missing-Value Detection and Imputation

**Name:** Madhura Vilas Suroshe  
**Course:** R Programming  
**Branch:** Computer Engineering  
**Semester:** VI  
**College:** Vidyalankar Institute of Technology, Mumbai

# Lab 3: Control Flow for Data Cleaning

## Problem Statement

A healthcare analytics team is analyzing patient cardiovascular data.
Before statistical analysis, the dataset must be checked and cleaned because
clinical data collected from different sources may contain invalid values,
missing observations, and extreme measurements.

For this practical, the resting blood pressure (`trestbps`) variable is used.
Artificial negative, missing, and extreme blood-pressure values are introduced
to simulate realistic data-entry problems.

### Objectives

1. Create a BP-cleaning function using if-else.
2. Implement error handling using tryCatch().
3. Compare loop-based and vectorized data cleaning.
4. Measure execution time.
5. Validate the cleaned dataset.

1. Install and Load Required Packages

In [ ]:
# Install required packages
install.packages(c("readr", "dplyr", "microbenchmark"),
                 repos = "https://cloud.r-project.org")

# Load packages
library(readr)
library(dplyr)
library(microbenchmark)

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



## 2. Loading the UCI Heart Disease Dataset

The UCI Heart Disease dataset contains clinical variables such as age,
resting blood pressure (`trestbps`), cholesterol, maximum heart rate, etc.

The original Cleveland dataset does not contain column names, so the required
column names are assigned manually.

In [ ]:
# URL of UCI Heart Disease Cleveland dataset
heart_url <- paste0(
  "https://archive.ics.uci.edu/ml/machine-learning-databases/",
  "heart-disease/processed.cleveland.data"
)

# Column names
heart_columns <- c(
  "age",
  "sex",
  "cp",
  "trestbps",
  "chol",
  "fbs",
  "restecg",
  "thalach",
  "exang",
  "oldpeak",
  "slope",
  "ca",
  "thal",
  "target"
)

# Read dataset
heart_data <- read.csv(
  heart_url,
  header = FALSE,
  col.names = heart_columns,
  na.strings = "?"
)

# Display first few records
head(heart_data)

# Dataset dimensions
dim(heart_data)

# Structure
str(heart_data)

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>
1,63,1,1,145,233,1,2,150,0,2.3,3,0,6,0
2,67,1,4,160,286,0,2,108,1,1.5,2,3,3,2
3,67,1,4,120,229,0,2,129,1,2.6,2,2,7,1
4,37,1,3,130,250,0,0,187,0,3.5,3,0,3,0
5,41,0,2,130,204,0,2,172,0,1.4,1,0,3,0
6,56,1,2,120,236,0,0,178,0,0.8,1,0,3,0


[1] 303  14

'data.frame':	303 obs. of  14 variables:
 $ age     : num  63 67 67 37 41 56 62 57 63 53 ...
 $ sex     : num  1 1 1 1 0 1 0 0 1 1 ...
 $ cp      : num  1 4 4 3 2 2 4 4 4 4 ...
 $ trestbps: num  145 160 120 130 130 120 140 120 130 140 ...
 $ chol    : num  233 286 229 250 204 236 268 354 254 203 ...
 $ fbs     : num  1 0 0 0 0 0 0 0 0 1 ...
 $ restecg : num  2 2 2 0 2 0 2 0 2 2 ...
 $ thalach : num  150 108 129 187 172 178 160 163 147 155 ...
 $ exang   : num  0 1 1 0 0 0 0 1 0 1 ...
 $ oldpeak : num  2.3 1.5 2.6 3.5 1.4 0.8 3.6 0.6 1.4 3.1 ...
 $ slope   : num  3 2 2 3 1 1 3 1 2 3 ...
 $ ca      : num  0 3 2 0 0 0 2 0 1 0 ...
 $ thal    : num  6 3 7 3 3 3 3 3 7 7 ...
 $ target  : int  0 2 1 0 0 0 3 0 2 1 ...


# Summary of resting blood pressure
summary(heart_data$trestbps)

# Check missing values
sum(is.na(heart_data$trestbps))

# Minimum and maximum BP
min(heart_data$trestbps, na.rm = TRUE)
max(heart_data$trestbps, na.rm = TRUE)

## 4. Introducing Invalid and Extreme BP Values

To simulate real-world data-entry problems, a few invalid values are
deliberately introduced:

- Negative BP values
- Missing (`NA`) values
- BP values greater than 300 mmHg

These artificial errors will later be detected and cleaned.

In [ ]:
# Set seed for reproducibility
set.seed(123)

# Copy original data
heart_dirty <- heart_data

# Select random positions
bp_indices <- sample(1:nrow(heart_dirty), 6)

# Introduce negative BP values
heart_dirty$trestbps[bp_indices[1:2]] <- c(-120, -80)

# Introduce missing values
heart_dirty$trestbps[bp_indices[3:4]] <- NA

# Introduce extreme BP values
heart_dirty$trestbps[bp_indices[5:6]] <- c(320, 350)

# Display affected rows
heart_dirty[bp_indices, c("age", "trestbps", "chol")]

,age,trestbps,chol
,<dbl>,<dbl>,<dbl>
179,43,-120,315
14,44,-80,263
195,68,NA,211
118,35,NA,183
299,45,320,264
229,54,350,206


## 5. Custom BP Cleaning Function

The function follows these rules:

- If BP is negative, convert it to `NA`.
- If BP is greater than 250 mmHg, cap it at 250.
- If BP is valid, retain it without modification.
- If BP is already `NA`, retain it as `NA`.

In [ ]:
clean_bp <- function(bp) {

  if (is.na(bp)) {
    return(NA_real_)
  } else if (bp < 0) {
    return(NA_real_)
  } else if (bp > 250) {
    return(250)
  } else {
    return(bp)
  }
}

In [ ]:
# Test individual values

clean_bp(-100)
clean_bp(120)
clean_bp(320)
clean_bp(NA)

[1] NA

[1] 120

[1] 250

[1] NA

# Expected behavior:
# -100 -> NA
# 120  -> 120
# 320  -> 250
# NA   -> NA

In [ ]:
# Apply custom function to BP column
heart_clean <- heart_dirty

heart_clean$trestbps <- sapply(
  heart_clean$trestbps,
  clean_bp
)

# Display cleaned BP values
head(heart_clean[, c("age", "trestbps", "chol")])

,age,trestbps,chol
,<dbl>,<dbl>,<dbl>
1,63,145,233
2,67,160,286
3,67,120,229
4,37,130,250
5,41,130,204
6,56,120,236


# 8. Error Handling Using tryCatch()

`tryCatch()` is used to safely perform calculations without terminating
the complete program when an invalid value is encountered.

Two calculations are performed:

1. Mean resting blood pressure when missing values are present.
2. Cholesterol-to-BP ratio.

The ratio calculation must handle cases where BP is zero, `NA`, negative,
or otherwise invalid.

In [ ]:
safe_mean_bp <- function(bp) {

  tryCatch(
    {
      if (length(bp) == 0) {
        stop("BP vector is empty.")
      }

      if (all(is.na(bp))) {
        stop("All BP values are missing.")
      }

      mean(bp, na.rm = TRUE)
    },

    error = function(e) {
      message("Error while calculating mean BP: ", e$message)
      return(NA_real_)
    },

    warning = function(w) {
      message("Warning while calculating mean BP: ", w$message)
      invokeRestart("muffleWarning")
    }
  )
}

In [ ]:
# Calculate mean BP
mean_bp <- safe_mean_bp(heart_clean$trestbps)

mean_bp

[1] 132.689

In [ ]:
safe_ratio <- function(chol, bp) {

  tryCatch(
    {
      if (is.na(chol) || is.na(bp)) {
        stop("Cholesterol or BP is missing.")
      }

      if (bp <= 0) {
        stop("BP denominator must be greater than zero.")
      }

      if (!is.finite(chol) || !is.finite(bp)) {
        stop("Invalid numeric value detected.")
      }

      chol / bp
    },

    error = function(e) {
      message("Ratio calculation error: ", e$message)
      return(NA_real_)
    }
  )
}

In [ ]:
# Test ratio with valid values
safe_ratio(200, 120)

# Test with zero denominator
safe_ratio(200, 0)

# Test with missing denominator
safe_ratio(200, NA)

# Test with negative denominator
safe_ratio(200, -100)

[1] 1.666667

Ratio calculation error: BP denominator must be greater than zero.



[1] NA

Ratio calculation error: Cholesterol or BP is missing.



[1] NA

Ratio calculation error: BP denominator must be greater than zero.



[1] NA

In [ ]:
heart_clean$chol_trestbps_ratio <- mapply(
  safe_ratio,
  heart_clean$chol,
  heart_clean$trestbps
)

# Display result
head(
  heart_clean[, c(
    "chol",
    "trestbps",
    "chol_trestbps_ratio"
  )]
)

Ratio calculation error: Cholesterol or BP is missing.

Ratio calculation error: Cholesterol or BP is missing.

Ratio calculation error: Cholesterol or BP is missing.

Ratio calculation error: Cholesterol or BP is missing.



,chol,trestbps,chol_trestbps_ratio
,<dbl>,<dbl>,<dbl>
1,233,145,1.606897
2,286,160,1.787500
3,229,120,1.908333
4,250,130,1.923077
5,204,130,1.569231
6,236,120,1.966667


# 12. Loop-Based Data Cleaning

A `for` loop is used to identify invalid and outlier BP values.

The loop checks every observation individually:

- Negative BP → `NA`
- BP > 250 → 250
- Valid BP → unchanged

In [ ]:
loop_clean_bp <- function(bp_vector) {

  result <- bp_vector

  for (i in seq_along(result)) {

    if (is.na(result[i])) {
      result[i] <- NA_real_

    } else if (result[i] < 0) {
      result[i] <- NA_real_

    } else if (result[i] > 250) {
      result[i] <- 250
    }
  }

  return(result)
}

# 13. Vectorized Data Cleaning

R provides vectorized operations that can process an entire vector without
explicitly iterating through each observation using a `for` loop.

In [ ]:
vectorized_clean_bp <- function(bp_vector) {

  result <- bp_vector

  # Convert negative values to NA
  result[result < 0] <- NA

  # Cap values greater than 250
  result[result > 250] <- 250

  return(result)
}

In [ ]:
loop_result <- loop_clean_bp(heart_dirty$trestbps)

vector_result <- vectorized_clean_bp(heart_dirty$trestbps)

# Check whether both methods produce the same result
identical(loop_result, vector_result)

[1] TRUE

# 15. Execution Time Comparison

The execution time of the loop-based and vectorized approaches is measured
using `system.time()`.

Vectorized operations are generally expected to be faster because they
reduce explicit R-level iteration.

In [ ]:
# Loop execution time
loop_time <- system.time(
  loop_clean_bp(heart_dirty$trestbps)
)

# Vectorized execution time
vector_time <- system.time(
  vectorized_clean_bp(heart_dirty$trestbps)
)

loop_time
vector_time

   user  system elapsed 
      0       0       0 

   user  system elapsed 
  0.004   0.000   0.004 

In [ ]:
# Create a larger BP vector for benchmarking
set.seed(123)

large_bp <- sample(
  c(
    runif(100000, 80, 220),
    runif(1000, -150, -1),
    runif(1000, 251, 350),
    NA
  ),
  200000,
  replace = TRUE
)

# Benchmark both approaches
benchmark_result <- microbenchmark(

  Loop = loop_clean_bp(large_bp),

  Vectorized = vectorized_clean_bp(large_bp),

  times = 10
)

benchmark_result

expr,time
<fct>,<dbl>
Vectorized,2256608
Vectorized,2031096
Loop,50073611
Vectorized,2161437
Vectorized,2065985
Loop,52516566
Loop,52523664
Vectorized,2106958
Loop,51730869


In [ ]:
summary(benchmark_result)

expr,min,lq,mean,median,uq,max,neval
<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
Loop,47.927032,50.393974,51.00930,51.312534,51.730869,52.523664,10
Vectorized,2.030279,2.065985,2.11536,2.117224,2.151494,2.256608,10


In [ ]:
# Compare median execution time
aggregate(time ~ expr, benchmark_result, median)

expr,time
<fct>,<dbl>
Loop,51312534
Vectorized,2117224


# 18. Validation of Cleaned Data

The cleaned dataset is validated by calculating:

- Number of missing BP values
- Minimum BP
- Maximum BP
- Mean BP
- Median BP
- Number of negative BP values
- Number of BP values greater than 250

In [ ]:
# Number of missing BP values
missing_bp <- sum(is.na(heart_clean$trestbps))

# Minimum BP
min_bp <- min(heart_clean$trestbps, na.rm = TRUE)

# Maximum BP
max_bp <- max(heart_clean$trestbps, na.rm = TRUE)

# Mean BP
mean_bp <- mean(heart_clean$trestbps, na.rm = TRUE)

# Median BP
median_bp <- median(heart_clean$trestbps, na.rm = TRUE)

# Negative values remaining
negative_bp <- sum(
  heart_clean$trestbps < 0,
  na.rm = TRUE
)

# Values above 250 remaining
above_250_bp <- sum(
  heart_clean$trestbps > 250,
  na.rm = TRUE
)

# Display validation results
validation_results <- data.frame(
  Metric = c(
    "Missing BP",
    "Minimum BP",
    "Maximum BP",
    "Mean BP",
    "Median BP",
    "Negative BP Values",
    "BP Values > 250"
  ),

  Value = c(
    missing_bp,
    min_bp,
    max_bp,
    mean_bp,
    median_bp,
    negative_bp,
    above_250_bp
  )
)

validation_results

Metric,Value
<chr>,<dbl>
Missing BP,4.000
Minimum BP,94.000
Maximum BP,250.000
Mean BP,132.689
Median BP,130.000
Negative BP Values,0.000
BP Values > 250,0.000


In [ ]:
if (negative_bp == 0 && above_250_bp == 0) {
  print("Validation successful: No negative or >250 BP values remain.")
} else {
  print("Validation failed: Invalid BP values remain.")
}

[1] "Validation successful: No negative or >250 BP values remain."


In [ ]:
write.csv(
  heart_clean,
  "cleaned_heart_data.csv",
  row.names = FALSE
)

print("cleaned_heart_data.csv created successfully.")

[1] "cleaned_heart_data.csv created successfully."


In [ ]:
# Check exported file
file.exists("cleaned_heart_data.csv")

[1] TRUE

## Lab 3 Conclusion

The UCI Heart Disease dataset was successfully cleaned using conditional
statements and a custom BP-cleaning function.

Negative blood-pressure values were converted to `NA`, while values greater
than 250 mmHg were capped at 250. The `tryCatch()` mechanism was used to
handle missing and invalid values during mean and ratio calculations without
terminating the program.

Both loop-based and vectorized approaches produced the same cleaning result.
The execution-time comparison demonstrated that vectorized R operations are
generally more efficient than explicit `for` loops for this type of data
processing.

The cleaned dataset was validated to ensure that no negative BP values or
values greater than 250 remained.

# Lab 4: Advanced Missing Data Handling

## Problem Statement

An organization is analyzing demographic and employment data for workforce
analytics. However, the dataset contains incomplete and potentially invalid
observations.

The UCI Adult / Census Income dataset is used to demonstrate different forms
of missing and invalid data including:

- NA
- NaN
- Blank strings
- NULL
- Impossible numeric values such as age = 999

### Objectives

1. Identify different forms of missing and invalid data.
2. Develop an appropriate missing-data treatment strategy.
3. Create a custom median-imputation function.
4. Analyze missingness before and after cleaning.
5. Validate the cleaned dataset.

In [ ]:
install.packages(
  c("readr", "dplyr", "naniar", "skimr"),
  repos = "https://cloud.r-project.org"
)

library(readr)
library(dplyr)
library(naniar)
library(skimr)

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘gridExtra’, ‘plyr’, ‘norm’, ‘visdat’, ‘viridis’, ‘UpSetR’



Attaching package: ‘skimr’


The following object is masked from ‘package:naniar’:

    n_complete




## 22. Loading the UCI Adult Dataset

The Adult dataset contains demographic and employment-related attributes such
as age, work class, education, occupation, hours per week and income.

In [ ]:
adult_url <- paste0(
  "https://archive.ics.uci.edu/ml/machine-learning-databases/",
  "adult/adult.data"
)

adult_columns <- c(
  "age",
  "workclass",
  "fnlwgt",
  "education",
  "education_num",
  "marital_status",
  "occupation",
  "relationship",
  "race",
  "sex",
  "capital_gain",
  "capital_loss",
  "hours_per_week",
  "native_country",
  "income"
)

adult_data <- read.csv(
  adult_url,
  header = FALSE,
  col.names = adult_columns,
  na.strings = "?",
  strip.white = TRUE,
  stringsAsFactors = FALSE
)

head(adult_data)

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
,<int>,<chr>,<int>,<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<int>,<int>,<chr>,<chr>
1,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
2,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
3,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
4,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
5,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K
6,37,Private,284582,Masters,14,Married-civ-spouse,Exec-managerial,Wife,White,Female,0,0,40,United-States,<=50K


In [ ]:
# Dimensions
dim(adult_data)

# Structure
str(adult_data)

# Summary
summary(adult_data)

[1] 32561    15

'data.frame':	32561 obs. of  15 variables:
 $ age           : int  39 50 38 53 28 37 49 52 31 42 ...
 $ workclass     : chr  "State-gov" "Self-emp-not-inc" "Private" "Private" ...
 $ fnlwgt        : int  77516 83311 215646 234721 338409 284582 160187 209642 45781 159449 ...
 $ education     : chr  "Bachelors" "Bachelors" "HS-grad" "11th" ...
 $ education_num : int  13 13 9 7 13 14 5 9 14 13 ...
 $ marital_status: chr  "Never-married" "Married-civ-spouse" "Divorced" "Married-civ-spouse" ...
 $ occupation    : chr  "Adm-clerical" "Exec-managerial" "Handlers-cleaners" "Handlers-cleaners" ...
 $ relationship  : chr  "Not-in-family" "Husband" "Not-in-family" "Husband" ...
 $ race          : chr  "White" "White" "White" "Black" ...
 $ sex           : chr  "Male" "Male" "Male" "Male" ...
 $ capital_gain  : int  2174 0 0 0 0 0 0 0 14084 5178 ...
 $ capital_loss  : int  0 0 0 0 0 0 0 0 0 0 ...
 $ hours_per_week: int  40 13 40 40 40 40 16 45 50 40 ...
 $ native_country: chr  "United-States" "Uni

      age            workclass         fnlwgt            education    
 Min.   :17.00   Length   :32561   Min.   :  12285   Length   :32561  
 1st Qu.:28.00   N.unique :    8   1st Qu.: 117827   N.unique :   16  
 Median :37.00   N.blank  :    0   Median : 178356   N.blank  :    0  
 Mean   :38.58   Min.nchar:    7   Mean   : 189778   Min.nchar:    3  
 3rd Qu.:48.00   Max.nchar:   16   3rd Qu.: 237051   Max.nchar:   12  
 Max.   :90.00   NAs      : 1836   Max.   :1484705                    
 education_num     marital_status      occupation       relationship  
 Min.   : 1.00   Length   :32561   Length   :32561   Length   :32561  
 1st Qu.: 9.00   N.unique :    7   N.unique :   14   N.unique :    6  
 Median :10.00   N.blank  :    0   N.blank  :    0   N.blank  :    0  
 Mean   :10.08   Min.nchar:    7   Min.nchar:    5   Min.nchar:    4  
 3rd Qu.:12.00   Max.nchar:   21   Max.nchar:   17   Max.nchar:   14  
 Max.   :16.00                     NAs      : 1843                    
      

## 24. Introducing Different Missing Data Types

Artificial missing and invalid values are introduced to demonstrate:

- `NA`
- `NaN`
- Blank strings (`""`)
- Impossible age values (`999`)

`NULL` is demonstrated separately because NULL represents the absence of an
R object rather than a normal missing value stored inside a data-frame cell.

In [ ]:
set.seed(123)

adult_dirty <- adult_data

# Select rows
missing_rows <- sample(1:nrow(adult_dirty), 20)

# Introduce NA in age
adult_dirty$age[missing_rows[1:5]] <- NA

# Introduce NaN in numeric variable
adult_dirty$hours_per_week[missing_rows[6:10]] <- NaN

# Introduce blank strings in categorical variable
adult_dirty$workclass[missing_rows[11:15]] <- ""

# Introduce impossible age values
adult_dirty$age[missing_rows[16:20]] <- 999

# Display affected records
adult_dirty[missing_rows, c(
  "age",
  "workclass",
  "hours_per_week",
  "education"
)]

,age,workclass,hours_per_week,education
,<dbl>,<chr>,<dbl>,<chr>
18847,NA,NA,40,Some-college
18895,NA,Private,40,HS-grad
26803,NA,Private,40,7th-8th
25102,NA,Self-emp-inc,55,Some-college
28867,NA,State-gov,37,Bachelors
2986,23,Private,NaN,Bachelors
1842,34,Local-gov,NaN,Some-college
25718,34,Self-emp-not-inc,NaN,HS-grad
3371,38,Private,NaN,Prof-school


In [ ]:
# Check NA values
is.na(adult_dirty$age)

# Count NA values
sum(is.na(adult_dirty$age))

[1] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
   [13] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
   [25] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
   [37] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
   [49] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
   [61] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
   [73] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
   [85] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
   [97] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [109] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [121] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [133] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [145] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [157] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [169] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [181] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [193] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [205] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [217] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [229] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [241] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [253] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [265] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [277] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [289] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [301] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [313] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [325] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [337] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [349] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [361] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [373] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [385] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [397] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [409] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [421] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [433] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [445] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [457] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [469] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [481] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [493] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [505] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [517] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [529] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [541] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [553] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [565] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [577] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [589] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [6

[1] 5

In [ ]:
# Check NaN values
is.nan(adult_dirty$hours_per_week)

# Count NaN values
sum(is.nan(adult_dirty$hours_per_week))

[1] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
   [13] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
   [25] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
   [37] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
   [49] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
   [61] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
   [73] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
   [85] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
   [97] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [109] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [121] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [133] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [145] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [157] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [169] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [181] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [193] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [205] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [217] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [229] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [241] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [253] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [265] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [277] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [289] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [301] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [313] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [325] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [337] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [349] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [361] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [373] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [385] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [397] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [409] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [421] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [433] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [445] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [457] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [469] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [481] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [493] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [505] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [517] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [529] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [541] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [553] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [565] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [577] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [589] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE
  [6

[1] 5

## Understanding NULL

`NULL` represents the absence of an R object.

It is different from `NA`, which represents a missing value inside a vector
or data frame.

A NULL object can be demonstrated as follows.

In [ ]:
# Create a NULL object
null_object <- NULL

# Check whether it is NULL
is.null(null_object)

[1] TRUE

In [ ]:
# Demonstration of NULL in a data frame

demo_df <- data.frame(
  name = c("A", "B", "C"),
  score = c(80, 90, 70)
)

# NULL normally means absence of an object.
# Assigning NULL to a data-frame column removes the column.

demo_df$temp <- c(1, 2, 3)

print(demo_df)

demo_df$temp <- NULL

print(demo_df)

  name score temp
1    A    80    1
2    B    90    2
3    C    70    3
  name score
1    A    80
2    B    90
3    C    70


In [ ]:
# Identify blank strings
blank_workclass <- adult_dirty$workclass == ""

# Count blank strings
sum(blank_workclass, na.rm = TRUE)

[1] 5

In [ ]:
# Identify age = 999
impossible_age <- adult_dirty$age == 999

# Count impossible values
sum(impossible_age, na.rm = TRUE)

[1] 5

In [ ]:
# Missing-value summary using naniar
miss_var_summary(adult_dirty)

variable,n_miss,pct_miss
<chr>,<int>,<num>
occupation,1843,5.66
workclass,1836,5.64
native_country,583,1.79
age,5,0.015[4m4[24m
hours_per_week,5,0.015[4m4[24m
fnlwgt,0,0
education,0,0
education_num,0,0
marital_status,0,0


# 31. Missingness Before Cleaning

The number and percentage of missing values are calculated before treatment.

In [ ]:
missing_before <- data.frame(
  Variable = names(adult_dirty),
  Missing_Count = sapply(
    adult_dirty,
    function(x) sum(is.na(x) | is.nan(x))
  )
)

missing_before$Missing_Percentage <- round(
  missing_before$Missing_Count /
    nrow(adult_dirty) * 100,
  2
)

missing_before

,Variable,Missing_Count,Missing_Percentage
,<chr>,<int>,<dbl>
age,age,5,0.02
workclass,workclass,1836,5.64
fnlwgt,fnlwgt,0,0.00
education,education,0,0.00
education_num,education_num,0,0.00
marital_status,marital_status,0,0.00
occupation,occupation,1843,5.66
relationship,relationship,0,0.00
race,race,0,0.00


In [ ]:
adult_clean <- adult_dirty

# Convert impossible age values to NA
adult_clean$age[
  !is.na(adult_clean$age) &
  adult_clean$age == 999
] <- NA

In [ ]:
# Replace blank workclass values with "Unknown"
adult_clean$workclass[
  adult_clean$workclass == ""
] <- "Unknown"

# Replace blank native_country values if any
adult_clean$native_country[
  adult_clean$native_country == ""
] <- "Unknown"

# Replace blank occupation values if any
adult_clean$occupation[
  adult_clean$occupation == ""
] <- "Unknown"

# 34. Custom Median Imputation Function

The custom function:

1. Accepts a numeric vector.
2. Identifies missing values.
3. Calculates the median of valid observations.
4. Replaces missing values with the median.
5. Returns the cleaned vector.

In [ ]:
median_impute <- function(x) {

  if (!is.numeric(x)) {
    stop("Input must be numeric.")
  }

  # Treat NaN as missing
  x[is.nan(x)] <- NA

  # Calculate median of valid observations
  valid_values <- x[!is.na(x)]

  if (length(valid_values) == 0) {
    warning("No valid values available for median imputation.")
    return(x)
  }

  med <- median(valid_values)

  # Replace missing values
  x[is.na(x)] <- med

  return(x)
}

In [ ]:
test_vector <- c(
  20,
  25,
  NA,
  30,
  NaN,
  35
)

print(test_vector)

imputed_vector <- median_impute(test_vector)

print(imputed_vector)

[1]  20  25  NA  30 NaN  35
[1] 20.0 25.0 27.5 30.0 27.5 35.0


In [ ]:
# Numeric variables selected for imputation
adult_clean$age <- median_impute(adult_clean$age)

adult_clean$hours_per_week <- median_impute(
  adult_clean$hours_per_week
)

adult_clean$fnlwgt <- median_impute(
  adult_clean$fnlwgt
)

In [ ]:
# Convert remaining NaN values to NA
adult_clean[] <- lapply(
  adult_clean,
  function(x) {
    if (is.numeric(x)) {
      x[is.nan(x)] <- NA
    }
    return(x)
  }
)

# Remove observations containing remaining NA values
adult_clean <- adult_clean[
  complete.cases(adult_clean),
]

In [ ]:
# Identify complete observations
complete_rows <- complete.cases(adult_clean)

# Count complete and incomplete observations
sum(complete_rows)
sum(!complete_rows)

[1] 30162

[1] 0

# 39. Missingness After Cleaning

The missing-value count and percentage are recalculated after treatment.
This allows the before-and-after data quality to be compared.

In [ ]:
missing_after <- data.frame(
  Variable = names(adult_clean),
  Missing_Count = sapply(
    adult_clean,
    function(x) sum(is.na(x) | is.nan(x))
  )
)

missing_after$Missing_Percentage <- round(
  missing_after$Missing_Count /
    nrow(adult_clean) * 100,
  2
)

missing_after

,Variable,Missing_Count,Missing_Percentage
,<chr>,<int>,<dbl>
age,age,0,0
workclass,workclass,0,0
fnlwgt,fnlwgt,0,0
education,education,0,0
education_num,education_num,0,0
marital_status,marital_status,0,0
occupation,occupation,0,0
relationship,relationship,0,0
race,race,0,0


In [ ]:
comparison <- data.frame(
  Variable = names(adult_dirty),

  Missing_Before = sapply(
    adult_dirty,
    function(x) sum(is.na(x) | is.nan(x))
  ),

  Missing_After = sapply(
    adult_clean,
    function(x) sum(is.na(x) | is.nan(x))
  )
)

comparison

,Variable,Missing_Before,Missing_After
,<chr>,<int>,<int>
age,age,5,0
workclass,workclass,1836,0
fnlwgt,fnlwgt,0,0
education,education,0,0
education_num,education_num,0,0
marital_status,marital_status,0,0
occupation,occupation,1843,0
relationship,relationship,0,0
race,race,0,0


# 41. Missingness Visualization

The `naniar` package is used to visualize the missingness pattern in the
dataset.

The visualization is generated before cleaning so that the missing-data
problem can be observed clearly.

In [ ]:
# Missingness visualization
vis_miss(adult_dirty)

In [ ]:
# Before cleaning
miss_var_summary(adult_dirty)

In [47]:
# After cleaning
miss_var_summary(adult_clean)

variable,n_miss,pct_miss
<chr>,<int>,<num>
age,0,0
workclass,0,0
fnlwgt,0,0
education,0,0
education_num,0,0
marital_status,0,0
occupation,0,0
relationship,0,0
race,0,0


# 44. Dataset Validation Using skimr

`skimr::skim()` provides a comprehensive summary of the cleaned dataset,
including data types, missing values, numerical summaries, and categorical
information.

In [48]:
skim(adult_clean)

,skim_type,skim_variable,n_missing,complete_rate,character.min,character.max,character.empty,character.n_unique,character.whitespace,numeric.mean,numeric.sd,numeric.p0,numeric.p25,numeric.p50,numeric.p75,numeric.p100,numeric.hist
,<chr>,<chr>,<int>,<dbl>,<int>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
1,character,workclass,0,1,7,16,0,8,0,NA,NA,NA,NA,NA,NA,NA,NA
2,character,education,0,1,3,12,0,16,0,NA,NA,NA,NA,NA,NA,NA,NA
3,character,marital_status,0,1,7,21,0,7,0,NA,NA,NA,NA,NA,NA,NA,NA
4,character,occupation,0,1,5,17,0,14,0,NA,NA,NA,NA,NA,NA,NA,NA
5,character,relationship,0,1,4,14,0,6,0,NA,NA,NA,NA,NA,NA,NA,NA
6,character,race,0,1,5,18,0,5,0,NA,NA,NA,NA,NA,NA,NA,NA
7,character,sex,0,1,4,6,0,2,0,NA,NA,NA,NA,NA,NA,NA,NA
8,character,native_country,0,1,4,26,0,41,0,NA,NA,NA,NA,NA,NA,NA,NA
9,character,income,0,1,4,5,0,2,0,NA,NA,NA,NA,NA,NA,NA,NA


In [49]:
# Check for age = 999
sum(adult_clean$age == 999, na.rm = TRUE)

[1] 0

In [50]:
# Check blank workclass values
sum(adult_clean$workclass == "", na.rm = TRUE)

[1] 0

In [51]:
# Check blank occupation values
sum(adult_clean$occupation == "", na.rm = TRUE)

[1] 0

In [52]:
# Check missing values in selected numeric variables

sum(is.na(adult_clean$age))
sum(is.na(adult_clean$hours_per_week))
sum(is.na(adult_clean$fnlwgt))

[1] 0

[1] 0

[1] 0

In [53]:
# Total missing values
total_missing <- sum(
  sapply(
    adult_clean,
    function(x) sum(is.na(x) | is.nan(x))
  )
)

# Impossible age values
invalid_age <- sum(
  adult_clean$age == 999,
  na.rm = TRUE
)

# Blank workclass values
blank_workclass_final <- sum(
  adult_clean$workclass == "",
  na.rm = TRUE
)

# Validation
if (
  total_missing == 0 &&
  invalid_age == 0 &&
  blank_workclass_final == 0
) {

  print("Validation successful: cleaned dataset contains no untreated missing values, impossible ages, or blank workclass values.")

} else {

  print("Validation requires further inspection.")
}

[1] "Validation successful: cleaned dataset contains no untreated missing values, impossible ages, or blank workclass values."


In [54]:
write.csv(
  adult_clean,
  "cleaned_adult_data.csv",
  row.names = FALSE
)

print("cleaned_adult_data.csv created successfully.")

[1] "cleaned_adult_data.csv created successfully."


In [55]:
file.exists("cleaned_adult_data.csv")

[1] TRUE

In [56]:
head(adult_clean)

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
,<dbl>,<chr>,<int>,<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<int>,<dbl>,<chr>,<chr>
1,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
2,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
3,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
4,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
5,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K
6,37,Private,284582,Masters,14,Married-civ-spouse,Exec-managerial,Wife,White,Female,0,0,40,United-States,<=50K


In [57]:
dim(adult_clean)

[1] 30162    15

## Lab 4 Conclusion

The UCI Adult dataset was successfully analyzed for different forms of
missing and invalid data.

The concepts of NA, NaN, NULL, and blank strings were demonstrated.
Impossible age values such as 999 were converted to NA, while blank
categorical values were replaced with "Unknown".

A custom median-imputation function was developed to replace missing numeric
values with the median of valid observations. The `complete.cases()` function
was used to identify complete observations, and remaining unrecoverable
missing observations were removed where appropriate.

The `naniar` package was used to summarize and visualize missingness, while
`skimr::skim()` was used to validate the cleaned dataset.

The comparison of missingness before and after cleaning showed that the data
quality improved after applying the missing-data treatment techniques.

# Overall Conclusion

This assignment provided practical experience in data cleaning and
preprocessing using R.

In Lab 3, conditional statements, custom functions, loops, vectorized
operations, and `tryCatch()` were applied to clean resting blood-pressure
data from the UCI Heart Disease dataset. Invalid negative values were
converted to NA and extreme values were capped at 250 mmHg. Loop-based and
vectorized implementations were benchmarked, demonstrating the efficiency
of vectorized R operations.

In Lab 4, different forms of missing data including NA, NaN, NULL, and blank
strings were studied using the UCI Adult dataset. Invalid values were
identified and treated using appropriate techniques. A custom median
imputation function was implemented for numeric variables, while blank
categorical values were replaced with "Unknown".

The cleaned datasets were validated using built-in R functions and packages
such as naniar and skimr. The experiments demonstrate the importance of
systematic data cleaning, error handling, validation, and efficient
programming before performing statistical analysis or machine learning.

# Final Deliverables

The following files were generated during this notebook:

1. `cleaned_heart_data.csv`
2. `cleaned_adult_data.csv`

The notebook also contains:

- Custom BP-cleaning function
- Custom median-imputation function
- `tryCatch()` error-handling functions
- Loop-based cleaning
- Vectorized cleaning
- Execution-time comparison
- Missing-data summaries
- Missingness visualizations
- Dataset validation
- `skimr::skim()` output
- Conclusions and interpretation

In [58]:
# Confirm that both files exist

print("Heart Disease CSV:")
print(file.exists("cleaned_heart_data.csv"))

print("Adult Dataset CSV:")
print(file.exists("cleaned_adult_data.csv"))

[1] "Heart Disease CSV:"
[1] TRUE
[1] "Adult Dataset CSV:"
[1] TRUE


In [59]:
# Download files in Google Colab

download.file(
  "file:///content/cleaned_heart_data.csv",
  "cleaned_heart_data.csv"
)

download.file(
  "file:///content/cleaned_adult_data.csv",
  "cleaned_adult_data.csv"
)